In [45]:
import cobra

import pandas as pd

from Bio.Seq import Seq
from Bio.Alphabet import generic_dna

import requests, sys, json, re
sys.path.insert(1, '../') # comment out in python script
from load_environmental_variables import *

In [ ]:
# #load and parsefiles


# # models--------
# human_model = cobra.io.load_json_model(local_data_path + 'raw/RECON3D.json')
# with open(local_data_path + 'raw/Recon3D_genes.txt') as f:
#     d = json.load(f)
# bigg_map = dict()
# for i in d['results']:
#         bigg_map[i['bigg_id']] = i['name']

human_model = cobra.io.load_matlab_model(root_path + 'MammalianSecretoryRecon/MODELS/RECON2_2.mat')
human_model_2 = cobra.io.load_json_model(local_data_path + 'raw/RECON3D.json')

# two genes in recon2_2 have HGNC:HGNC:### rather than HGNC:###, the following code corrects that issue

# since the two genes are involved in the same three reactions, simply need to rewrite these three reactions
# rather than looping through
genes_to_duplicate = [gene.id for gene in human_model.genes if gene.id.count(':') > 1]
g0, g1 = genes_to_duplicate[0], genes_to_duplicate[1]  
g_i = human_model.genes.get_by_id(g0)
r_i = list(g_i.reactions)
g_c = human_model.genes.get_by_id(g0[5:])

# the following line of code gets rid of both genes in genes to duplicate since they are incolved in the same
# reations
human_model.remove_reactions(r_i, remove_orphans=True)
for r in r_i:
    r0 = r.gene_reaction_rule.replace(g0, g0[5:])
    r.gene_reaction_rule = r0.replace(g1, g1[5:])
cobra.io.save_json_model(human_model, local_data_path + 'processed/corrected_recon2_2.json')

compartments = human_model_2.compartments
compartments['pm'] = 'plasma membrane'

In [581]:
# load and parse files

# # protein information--------
psim_human = pd.read_csv(root_path + 'MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/PSIM_HUMAN.tab', 
                         sep = '\t')



identifiers = pd.read_csv(local_data_path + 'raw/identifiers.txt', sep = '\t')
idx = identifiers['NCBI gene ID'].dropna().index
new_vals = identifiers['NCBI gene ID'].dropna().astype(int).astype(str)
identifiers.loc[idx, 'NCBI gene ID'] = new_vals

# # machinery lists-----------
# get all genes in human + secretory model
# get the corresponding premrna, mrna, protein seq, location, and ptms

# get machinery for each module: machinery (metabolic, secretory, expression) and secreted proteins

# metabolic_machinery in HGNC format
metabolic_machinery = sorted(set([gene.id for gene in human_model.genes]))

# secretory machinery in NCBI gene format, convert to HGNC 
rxnGPR = open(root_path + "MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/rxnGPRs_HUMAN.txt").read().splitlines()[1:]
secretory_machinery = []
for i in rxnGPR:
    secretory_machinery += re.findall(r'\d+', i)
secretory_machinery = sorted(set(secretory_machinery))
ncbi_map = dict(zip(identifiers['NCBI gene ID'].tolist(), identifiers['HGNC ID'].tolist()))
secretory_machinery = [ncbi_map[i] for i in secretory_machinery]

# expression machinery

# secreted proteins - anything with a signal peptide for now
secP = psim_human[psim_human.SP == 1]['Entry'].tolist()
secP = sorted(set(secP).intersection(identifiers['UniProt accession'].dropna().tolist()))
secP  = identifiers[identifiers['UniProt accession'].isin(secP)]['HGNC ID'].tolist()


proteins = sorted(set(metabolic_machinery + secretory_machinery + secP))  

Generate PSIM for ME model based off of gene lists

Some issues here: 
    1) gene_sequence length and length from info variable (commented out) show slight discrepancies
    2) not 100% sure of the transcribe() function being appropriate
    3) I’m taking the first transcript from entrez api

In [602]:
# hyperlink = "https://rest.ensembl.org/lookup/id/" + identifiers.loc[g, 'Ensembl gene ID'] + '?'
# info = requests.get(hyperlink+'expand=1', headers={ "Content-Type" : "application/json"}).json()
def get_premrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?' 
        gene_sequence = requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text
        return str(Seq(gene_sequence).transcribe()) # introns and UTRs
    except:
        return float('nan')

def get_mrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?' 
        cdna = requests.get(hyperlink+'type=cdna;multiple_sequences=2', 
                            headers={ "Content-Type" : "text/plain"}).text.splitlines()[0]
        return str(Seq(cdna, generic_dna).transcribe()) # UTRs, no introns, no polyA tail right now 
    except:
        return float('nan')

def get_protein_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?' 
        return requests.get(hyperlink+'type=protein;multiple_sequences=2', 
                            headers={ "Content-Type" : "text/plain"}).text.splitlines()[0]
    except:
        return float('nan')

In [ ]:
cols = ['HGNC ID', 'Ensembl gene ID', 'UniProt accession', 'NCBI gene ID']
idx = identifiers['HGNC ID'].isin(proteins)
psim_me = identifiers.loc[idx, cols]
psim_me.reset_index(inplace = True, drop = True)
psim_me['premrna_seq'] = psim_me['Ensembl gene ID'].apply(get_premrna_seq)
psim_me['mrna_seq'] = psim_me['Ensembl gene ID'].apply(get_mrna_seq)
psim_me['protein_seq'] = psim_me['Ensembl gene ID'].apply(get_protein_seq)

for col in psim_human.columns.tolist()[4:-1]:
    mapper = dict(zip(psim_human.Entry.tolist(), psim_human[col].tolist()))
    psim_me[col] = psim_me['UniProt accession'].map(mapper)

In [ ]:
metabolic_idx = psim_me[psim_me['HGNC ID'].isin(metabolic_machinery)].index.tolist()
secM_idx = psim_me[psim_me['HGNC ID'].isin(secretory_machinery)].index.tolist()
secP_idx = psim_me[psim_me['HGNC ID'].isin(secP)].index.tolist()
modules = dict()

for i in metabolic_idx:
    modules[i] = ['Metabolic Machinery']
for i in secM_idx:
    if i not in modules.keys():
        modules[i] = ['Secretory Machinery']
    else:
        modules[i] += ['Secretory Machinery']
for i in secP_idx:
    if i not in modules.keys():
        modules[i] = ['Secreted Protein']
    else:
        modules[i] += ['Secreted Protein']

# expression machinery

psim_me['Modules'] = psim_me.index.map(modules)

In [ ]:
2

In [ ]:
psim_me.to_csv(local_data_path + 'processed/psim_me.csv')